# Stage 2: Vector Search + LLM (Retrieval-Augmented Generation)

In Stage 1, we saw the LLM confidently invent citations that don't exist.
To fix this, we will **give it real papers to reference.** This is precisely Retrieval-Augmented Generation (or RAG): retrieve first, then generate.

In this notebook we'll:
1. Convert PubMed abstracts into vector embeddings
2. Store them in Qdrant (a vector database)
3. Retrieve relevant papers for a question
4. Feed those papers to the LLM as context
5. Discover the limits of pure vector search

## 1. Setup + Load Data

In [ ]:
import os
import json

from dotenv import load_dotenv
from openai import OpenAI

from qdrant_client import QdrantClient
from qdrant_client.models import models

load_dotenv("../.env")

client = OpenAI()
qdrant = QdrantClient(url=os.getenv("QDRANT_URL", "http://localhost:6333"))

COLLECTION = "pubmed_dense"
EMBED_MODEL = "text-embedding-3-small"
EMBED_DIM = 1536

with open("../data/pubmed_dataset.json") as f:
    dataset = json.load(f)

# filter papers with abstracts, limit to 5,000 for demo
papers = [p for p in dataset["papers"] if p.get("abstract")][:5_000] 
print(f"Loaded {len(papers)} papers with abstracts")

Loaded 9762 papers with abstracts


## 2. What Are Vector Embeddings?

An embedding model converts text into a high-dimensional vector (a list of
numbers) where **similar meanings are close together** in the vector space.
OpenAI's `text-embedding-3-small` maps any text to a **1536-dimensional
vector**. Two abstracts about the same disease will end up near each other;
an abstract about oncology and one about baking will be far apart.

This is the core idea behind semantic search: instead of matching keywords,
we match *meaning*.

## 3. Create Qdrant Collection

In [11]:
# Delete if exists, create fresh
if qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=models.VectorParams(
        size=EMBED_DIM,
        distance=models.Distance.COSINE,
    ),
)
print(f"Collection '{COLLECTION}' created")

Collection 'pubmed_dense' created


## 4. Embed and Ingest Papers

We'll send abstracts to the OpenAI embedding API in batches of 100, then
upsert the resulting vectors (along with paper metadata) into Qdrant.

In [13]:
import time

from tqdm import tqdm

BATCH_SIZE = 100
MAX_TOKENS = 8191  # for text-embedding-3-small
start = time.time()

for i in tqdm(range(0, len(papers), BATCH_SIZE), desc="Ingesting papers"):
    batch = papers[i : i + BATCH_SIZE]
    texts = [p["abstract"][:MAX_TOKENS] for p in batch]

    # batch embed
    response = client.embeddings.create(model=EMBED_MODEL, input=texts)
    vectors = [e.embedding for e in response.data]

    # create points
    points = [
        models.PointStruct(
            id=int(paper["pmid"]),
            vector=vector,
            payload={
                "pmid": paper["pmid"],
                "title": paper["title"],
                "abstract": paper["abstract"][:MAX_TOKENS],
                "authors": [a["name"] for a in paper.get("authors", [])],
                "journal": paper.get("journal", ""),
                "mesh_terms": [m["term"] for m in paper.get("mesh_terms", [])],
            },
        )
        for paper, vector in zip(batch, vectors)
    ]

    qdrant.upsert(collection_name=COLLECTION, points=points)
    print(f"  Ingested {min(i + BATCH_SIZE, len(papers))}/{len(papers)} papers")

elapsed = time.time() - start
print(f"\nDone! {len(papers)} papers ingested in {elapsed:.1f}s")

Ingesting papers:   1%|          | 1/98 [00:01<01:58,  1.22s/it]

  Ingested 100/9762 papers


Ingesting papers:   2%|▏         | 2/98 [00:02<02:24,  1.51s/it]

  Ingested 200/9762 papers


Ingesting papers:   3%|▎         | 3/98 [00:03<02:02,  1.29s/it]

...

  Ingested 7500/9762 papers


Ingesting papers:  78%|███████▊  | 76/98 [01:41<00:49,  2.23s/it]

  Ingested 7600/9762 papers


Ingesting papers:  78%|███████▊  | 76/98 [01:42<00:29,  1.35s/it]


BadRequestError: Error code: 400 - {'error': {'message': "We could not parse the JSON body of your request. (HINT: This likely means you aren't using your HTTP library correctly. The OpenAI API expects a JSON payload, but what was sent was not valid JSON. If you have trouble figuring out how to fix this, please contact us through our help center at help.openai.com.)", 'type': 'invalid_request_error', 'param': None, 'code': None}}

## 5. Semantic Search

Now let's search! We embed our question into the same vector space and find
the nearest papers. Papers whose abstracts are semantically close to the
question will have the highest cosine similarity scores.

In [14]:
def search_papers(question: str, top_k: int = 5):
    """
    Embed the question and search Qdrant for similar papers.
    """
    q_embedding = client.embeddings.create(
        model=EMBED_MODEL, input=question
    ).data[0].embedding

    results = qdrant.query_points(
        collection_name=COLLECTION,
        query=q_embedding,
        limit=top_k,
        with_payload=True,
    )
    return results.points


# Search!
question = "What are the latest advances in using CRISPR-Cas9 for treating sickle cell disease?"
results = search_papers(question)

print(f"Query: {question}\n")
for i, point in enumerate(results, 1):
    print(f"{i}. [Score: {point.score:.4f}] {point.payload['title'][:90]}")
    print(f"   PMID: {point.payload['pmid']} | Journal: {point.payload['journal']}")
    print()

Query: What are the latest advances in using CRISPR-Cas9 for treating sickle cell disease?

1. [Score: 0.7530] Curing Sickle Cell Disease by Allogeneic Hematopoietic Stem Cell (HSC) Transplantation Tow
   PMID: 41300819 | Journal: Genes

2. [Score: 0.7441] Use of genome-editing tools to treat sickle cell disease.
   PMID: 27250347 | Journal: Human genetics

3. [Score: 0.7440] CRISPR/Cas9 System as a Promising Therapy in Thalassemia and Sickle Cell Disease: A System
   PMID: 39794549 | Journal: Molecular biotechnology

4. [Score: 0.7362] CRISPR/Cas9 in the treatment of sickle cell disease (SCD) and its comparison with traditio
   PMID: 39359808 | Journal: Annals of medicine and surgery (2012)

5. [Score: 0.7305] A justice-based argument for including sickle cell disease in CRISPR/Cas9 clinical researc
   PMID: 31107563 | Journal: Bioethics



## 6. RAG: Feed Context to the LLM

Here's the magic. Instead of asking the LLM to answer from its training
data (where it hallucinates), we **stuff the retrieved papers into the
prompt** and instruct it to answer using *only* that context.

In [15]:
def rag_answer(question: str, results: list[models.ScoredPoint]):
    """
    Generate an answer using retrieved papers as context.
    """
    context = "\n\n".join(
        [
            f"Paper {i+1} (PMID: {r.payload['pmid']}):\n"
            f"Title: {r.payload['title']}\n"
            f"Abstract: {r.payload['abstract'][:500]}"
            for i, r in enumerate(results)
        ]
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a biomedical research assistant. Answer questions "
                    "using ONLY the provided paper context. Cite papers by PMID."
                ),
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}",
            },
        ],
        temperature=0.0,
    )
    return response.choices[0].message.content


answer = rag_answer(question, results)
print(answer)

Recent advances in using CRISPR-Cas9 for treating sickle cell disease (SCD) include its application as a promising gene editing tool that can modify the genetic mutations responsible for the disease. Clinical trials are underway to evaluate the efficacy and safety of the CRISPR/Cas9 system in treating SCD, demonstrating its potential as a revolutionary approach compared to traditional treatment methods (PMID: 39794549, PMID: 39359808, PMID: 31107563). The technology allows for precise modifications of the β-globin gene, which is implicated in SCD, and has shown substantial success in preclinical and early clinical settings (PMID: 27250347).


## 7. RAG Works!

Now every citation is **real** -- the LLM is grounded in actual papers from
our database. No more hallucinated PMIDs. This is a massive improvement
over vanilla prompting.

But let's try a more specific, multi-faceted query and see what happens...

## 8. The Semantic Drift Problem

In [17]:
specific_question = "What papers discuss p53 tumor suppressor gene mutations specifically in glioblastoma multiforme?"
results_specific = search_papers(specific_question)

print(f"Query: {specific_question}\n")
for i, point in enumerate(results_specific, 1):
    print(f"{i}. [Score: {point.score:.4f}] {point.payload['title'][:90]}")
    # abstract sample
    print(f"   Abstract: {point.payload['abstract'][:200]}...")
    mesh = point.payload.get("mesh_terms", [])[:5]
    print(f"   MeSH: {', '.join(mesh)}")
    print()

Query: What papers discuss p53 tumor suppressor gene mutations specifically in glioblastoma multiforme?

1. [Score: 0.5536] Genome-wide CRISPR screens identify novel regulators of wild-type and mutant p53 stability
   Abstract: Tumor suppressor p53 (TP53) is frequently mutated in cancer, often resulting not only in loss of its tumor-suppressive function but also acquisition of dominant-negative and even oncogenic gain-of-fun...
   MeSH: Tumor Suppressor Protein p53, Animals, Humans, Mice, Protein Stability

2. [Score: 0.5518] Unlocking glioblastoma: breakthroughs in molecular mechanisms and next-generation therapie
   Abstract: Glioblastoma (GB) remains the most aggressive primary brain tumor in adults, characterized by rapid progression, recurrence, and resistance to conventional therapies. Despite advancements in surgical ...
   MeSH: Humans, Glioblastoma, Brain Neoplasms, Immunotherapy, Molecular Targeted Therapy

3. [Score: 0.5505] In vivo Engineering of Chromosome 19 q-arm by Empl

## 9. The Gap
Even with a partial index, dense retrieval worked well for broad queries (e.g., CRISPR + sickle cell disease), returning clearly relevant papers with strong scores (~0.73–0.75).

But the failure mode shows up on specific queries:

> *“What papers discuss p53 tumor suppressor gene mutations specifically in glioblastoma multiforme?”*

Top results had lower scores (~0.53–0.55) and mixed relevance:
- some papers were about **p53** but not specifically GBM,
- others were about **glioblastoma** but not focused on **p53 mutation evidence**.

**The problem:** dense vector search optimizes semantic similarity, not exact term matching.  
**What we need next:** combine semantic retrieval with lexical precision (keyword/BM25) — i.e., **hybrid search** in Stage 3.

## Architecture

```
Stage 2 architecture:
    Question --embed--> [Qdrant: Dense Vectors] --top-k--> Papers --> [LLM] --> Answer

Problem: Semantic drift -- related but wrong papers

Next: Add keyword search (BM25) for precision
```